# Concrete QC project - Step 2 (why?)

Goal: find out WHY the concrete behaves the way we saw in Step 1c.

For each question we write a **hypothesis** (a possible reason), say what we would expect to see if it is true,
then test it with the data. A hypothesis can be supported, rejected, or "not sure".

Important: the data can show that two things move together. That does not always prove one causes the other.
We will say this where it matters.

| # | Question from Step 1c | Hypothesis we test |
|---|---|---|
| H1 | Why are the low-strength mixes so erratic? | They go into many different small elements |
| H2 | Is "too wet" concrete weaker? | Water is added on site, so wet concrete should be weaker and lighter |
| H3 | Why is the slump often too high? | Some elements need wetter concrete |
| H4 | Why did 30/20 and 40/20 drop? | The elements changed over time, not the concrete |
| H5 | Does the season matter? | Hot summers and cold winters change the strength |
| H6 | Does fly ash slow early strength? | Fly ash mixes gain strength slower than the CRCP |
| H7 | Is stronger concrete more durable? | Pours with higher 28-day strength have better OPI and sorptivity |
| H8 | Why is the CRCP denser? | Its mix design is heavier |

### Step 1: Load the clean files

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

project = Path.cwd().parent   # the notebook runs from the notebooks folder, so the project is one level up
processed = project / "data" / "processed"
figures = project / "reports" / "figures"

cubes = pd.read_csv(processed / "cubes_clean.csv")
mixes = pd.read_csv(processed / "mix_designs.csv")
durability = pd.read_csv(processed / "durability_clean.csv")

cubes["date_cast"] = pd.to_datetime(cubes["date_cast"])
durability["date_cast"] = pd.to_datetime(durability["date_cast"])

# the element is the last part of the folder: "03_Structures/Ramp A/Piers" -> "Piers"
cubes["element_group"] = cubes["folder"].str.split("/").str[-1]

day7 = cubes[cubes["age_group"] == "7 days"].copy()
day28 = cubes[cubes["age_group"] == "28 days"].copy()
print("28-day results:", len(day28))

### How to read a p-value (we use it in H2 and H6)

When we compare two groups (for example wet vs normal concrete), the **p-value** tells us how likely it is that
the difference is just luck.
- p below 0.05 -> the difference is probably real
- p above 0.05 -> it could just be luck, we can not say there is a real difference

## H1 - Why are the low-strength mixes so erratic?

**Hypothesis:** 15/20, 20/20, 25/20 and 35/20 are used in many different small jobs
(stormwater, duct encasement, sidewalks ...). Different elements = different handling, so the results spread.

**If true:** inside ONE element the spread (cov) should be much smaller than for the whole mix.

In [ ]:
def spread_table(data):
    table = data.groupby("element_group")["strength"].agg(["count", "mean", "std"])
    table["cov_percent"] = table["std"] / table["mean"] * 100
    return table[table["count"] >= 5].sort_values("count", ascending=False).round(1)

for mix in ["AFR-20/20", "AFR-25/20", "AFR-35/20"]:
    one_mix = day28[day28["mix_id"] == mix]
    whole_cov = one_mix["strength"].std() / one_mix["strength"].mean() * 100
    print(mix, "- cov for the whole mix:", round(whole_cov, 1), "%")
    print(spread_table(one_mix))
    print()

**Second idea:** the report only says "20MPa" with no stone size (match = "class only").
Maybe some of those were really a different mix.

In [ ]:
low_mixes = day28[day28["mix_id"].isin(["AFR-15/20", "AFR-20/20", "AFR-25/20", "AFR-35/20"])]
print(low_mixes.groupby(["mix_id", "match"])["strength"].agg(["count", "mean", "std"]).round(1))

In [ ]:
# Picture: 20/20 strength in its main elements
mix = day28[day28["mix_id"] == "AFR-20/20"]
groups = mix["element_group"].value_counts()
groups = groups[groups >= 9].index

plt.figure(figsize=(9, 4))
plt.boxplot([mix.loc[mix["element_group"] == g, "strength"] for g in groups])
plt.axhline(20, color="red", label="class 20 MPa")
plt.xticks(range(1, len(groups) + 1), groups, rotation=20)
plt.ylabel("28-day strength (MPa)")
plt.title("20/20 - 28-day strength per element")
plt.legend()
plt.tight_layout()
plt.savefig(figures / "07_H1_20MPa_per_element.png", dpi=150)
plt.show()

**Your conclusion for H1:** Does the cov drop a lot inside one element? If it stays high (above 25-30%)
even inside one element, then the element is NOT the main reason, and the concrete itself (or how it is
sampled and cured on small jobs) is erratic.

## H2 - Is "too wet" concrete weaker?

**Hypothesis:** when the slump is above the design range, water was added on site.

**If true:** for the same mix, the "above" results should have LOWER strength and LOWER density than the "within" results.

We use `strength_vs_class` (strength / class) so we can compare mixes fairly.

In [ ]:
compare = day28[day28["slump_check"].isin(["within", "above"])]
table = compare.groupby(["mix_id", "slump_check"])[["strength_vs_class", "density"]].mean().unstack().round(2)
print(table)

In [ ]:
# Test it for the mixes that have enough results in both groups
for mix in ["AFR-20/20", "AFR-25/20", "AFR-30/20", "AFR-40/20", "AFR-35/28CRCP"]:
    one = compare[compare["mix_id"] == mix]
    wet = one.loc[one["slump_check"] == "above", "strength"]
    normal = one.loc[one["slump_check"] == "within", "strength"]
    t, p = stats.ttest_ind(wet, normal, equal_var=False)
    print(f"{mix}: wet mean {wet.mean():.1f} MPa ({len(wet)}), normal mean {normal.mean():.1f} MPa ({len(normal)}), p = {p:.3f}")

In [ ]:
# Picture: slump vs strength for 30/20
mix = day28[day28["mix_id"] == "AFR-30/20"]
plt.figure(figsize=(8, 4))
plt.scatter(mix["measured_slump"], mix["strength"], s=10, alpha=0.5)
plt.axvline(125, color="red", linestyle="--", label="design maximum 125 mm")
plt.xlabel("Measured slump (mm)")
plt.ylabel("28-day strength (MPa)")
plt.title("30/20 - slump vs 28-day strength")
plt.legend()
plt.tight_layout()
plt.savefig(figures / "08_H2_slump_vs_strength_30MPa.png", dpi=150)
plt.show()

print("Correlation slump vs strength (30/20):", round(mix["measured_slump"].corr(mix["strength"]), 2))

**Your conclusion for H2:** If the wet concrete is NOT weaker (or even a bit stronger), then the extra slump is
probably NOT from water added on site. A more likely reason is that the supplier sends the mix with more
admixture (plasticiser) so it is easier to place - which gives slump without losing strength.

## H3 - Why is the slump often too high?

**Hypothesis:** some elements (parapets, thin walls, lots of steel) need wetter concrete, so the contractor
asks for it.

**If true:** the % of "above" results should be much higher for some elements than others, for the same mix.

In [ ]:
for mix in ["AFR-30/20", "AFR-40/20"]:
    one = cubes[(cubes["mix_id"] == mix) & cubes["slump_check"].notna()]
    table = pd.crosstab(one["element_group"], one["slump_check"], normalize="index") * 100
    table["results"] = one["element_group"].value_counts()
    table = table[table["results"] >= 15].sort_values("above", ascending=False).round(0)
    print(mix)
    print(table)
    print()

**Also check:** has the slump changed over time? (median slump per year)

In [ ]:
print(cubes.pivot_table(index="year", columns="mix_id", values="measured_slump", aggfunc="median")[["AFR-30/20", "AFR-40/20", "AFR-W40/20", "AFR-35/28CRCP"]])

**Your conclusion for H3:** Is "too wet" linked to certain elements, or is it the same everywhere?
If it is the same everywhere, it points to how the mix is supplied rather than the element.

## H4 - Why did 30/20 and 40/20 drop over time?

**Hypothesis:** the concrete did not get weaker - the jobs changed. Early on 30/20 went into one type of
element and later into others with lower results.

**If true:** inside ONE element the strength should stay the same over time.

In [ ]:
day28_dated = day28.dropna(subset=["date_cast"]).copy()
day28_dated["half_year"] = day28_dated["date_cast"].dt.year.astype(str) + "-H" + ((day28_dated["date_cast"].dt.month > 6) + 1).astype(str)

mix = day28_dated[day28_dated["mix_id"] == "AFR-30/20"]

# Which elements did 30/20 go into, per half year?
print(pd.crosstab(mix["half_year"], mix["element_group"]).loc[:, lambda t: t.sum() >= 15])

In [ ]:
# The biggest element for 30/20 - did its strength drop too?
biggest = mix["element_group"].value_counts().index[0]
print("Biggest element:", biggest)
print(mix[mix["element_group"] == biggest].groupby("half_year")["strength"].agg(["count", "mean"]).round(1))

In [ ]:
# What else changed for 30/20 over time? Slump, density and the 7-day strength
summary = mix.groupby("half_year").agg(
    results=("strength", "count"),
    strength_28=("strength", "mean"),
    slump=("measured_slump", "median"),
    density=("density", "mean"),
).round(1)

seven = day7.dropna(subset=["date_cast"]).copy()
seven["half_year"] = seven["date_cast"].dt.year.astype(str) + "-H" + ((seven["date_cast"].dt.month > 6) + 1).astype(str)
summary["strength_7"] = seven[seven["mix_id"] == "AFR-30/20"].groupby("half_year")["strength"].mean().round(1)
print(summary)

In [ ]:
# Same check for 40/20
mix40 = day28_dated[day28_dated["mix_id"] == "AFR-40/20"]
print(pd.crosstab(mix40["half_year"], mix40["element_group"]).loc[:, lambda t: t.sum() >= 10])
print()
print(mix40.groupby("half_year").agg(results=("strength", "count"), strength_28=("strength", "mean"),
                                     slump=("measured_slump", "median"), density=("density", "mean")).round(1))

**Your conclusion for H4:** If the strength also dropped INSIDE the same element, the change is in the
concrete itself (materials, cement, plant) and not just in the type of job. Density staying the same but
strength dropping would point to the binder (cement / fly ash) rather than the aggregate.

## H5 - Does the season matter?

Cape Town: summer = Dec-Feb (hot), winter = Jun-Aug (cold and wet).

**Hypothesis:** hot weather makes the concrete stiffen faster and cure worse; cold weather slows down the
strength gain (especially with fly ash).

**If true:** 7-day strength should be lower in winter, and 28-day strength lower in summer.

In [ ]:
def season(month):
    if month in (12, 1, 2):
        return "summer"
    if month in (6, 7, 8):
        return "winter"
    return "autumn / spring"

for data, name in [(day7, "7-day"), (day28, "28-day")]:
    dated = data.dropna(subset=["date_cast"]).copy()
    dated["season"] = dated["date_cast"].dt.month.apply(season)
    print(name, "strength / class per season")
    print(dated.pivot_table(index="mix_id", columns="season", values="strength_vs_class", aggfunc="mean")
          .loc[["AFR-20/20", "AFR-30/20", "AFR-40/20", "AFR-W40/20", "AFR-W50/20"]].round(2))
    print()

**Your conclusion for H5:** Are the differences between seasons big (more than about 0.1) and the same
for most mixes? Or small and different from mix to mix? Small and mixed = season is not a main reason.

## H6 - Does fly ash slow down the early strength?

**Hypothesis:** fly ash reacts slower than cement, so fly ash mixes reach a smaller part of their 28-day
strength at 7 days.

**If true:** the 7/28 ratio of the fly ash mixes is clearly lower than the CRCP (no fly ash).

Careful: the CRCP is also a different element (road) and a different mix, so this is a fair first test, not proof.

In [ ]:
key = ["lr", "element", "date_cast", "mix_id"]
seven = day7.groupby(key)["strength"].mean().rename("strength_7")
twenty_eight = day28.groupby(key)["strength"].mean().rename("strength_28")
pairs = pd.concat([seven, twenty_eight], axis=1).dropna().reset_index()
pairs["ratio_7_28"] = pairs["strength_7"] / pairs["strength_28"]
pairs["fly_ash"] = pairs["mix_id"].apply(lambda m: "no fly ash" if "CRCP" in m else "30% fly ash")

print(pairs.groupby("fly_ash")["ratio_7_28"].agg(["count", "mean", "median"]).round(2))

with_ash = pairs.loc[pairs["fly_ash"] == "30% fly ash", "ratio_7_28"]
no_ash = pairs.loc[pairs["fly_ash"] == "no fly ash", "ratio_7_28"]
t, p = stats.ttest_ind(with_ash, no_ash, equal_var=False)
print("p-value:", round(p, 4))

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot([with_ash, no_ash])
plt.xticks([1, 2], ["30% fly ash (11 mixes)", "no fly ash (CRCP)"])
plt.ylabel("7-day / 28-day strength")
plt.title("Early strength: fly ash vs no fly ash")
plt.tight_layout()
plt.savefig(figures / "09_H6_fly_ash_ratio.png", dpi=150)
plt.show()

**Your conclusion for H6:** Is the difference big and is p below 0.05? What does it mean for judging
fly ash concrete at 7 days?

## H7 - Is stronger concrete more durable?

**Hypothesis:** pours with a higher 28-day cube strength also have a better OPI (higher) and sorptivity (lower).

To test this we join the durability results to the cube results of the SAME pour (same LR number).

In [ ]:
durability["lr_number"] = durability["lr"].str.extract(r"(\d+)").astype(int)
cubes["lr_number"] = cubes["lr"].str.extract(r"(\d+)").astype(float)

pour_strength = day28.assign(lr_number=day28["lr"].str.extract(r"(\d+)").astype(float)).groupby("lr_number").agg(
    strength_28=("strength", "mean"),
    slump=("measured_slump", "mean"),
    density=("density", "mean"),
    mix_id=("mix_id", "first"),
).reset_index()

joined = durability.merge(pour_strength, on="lr_number", how="inner")
print("Durability tests we could join to a 28-day cube result:", len(joined), "of", len(durability))
print(joined["mix_id"].value_counts())

In [ ]:
print("Correlation with OPI (higher = better):")
print(joined[["strength_28", "density", "slump", "age_days"]].corrwith(joined["opi_mean"]).round(2))
print()
print("Correlation with sorptivity (lower = better):")
print(joined[["strength_28", "density", "slump", "age_days"]].corrwith(joined["sorptivity_mean"]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(joined["strength_28"], joined["opi_mean"], s=15)
axes[0].axhline(9.3, color="red", linestyle="--")
axes[0].set_xlabel("28-day cube strength (MPa)")
axes[0].set_ylabel("OPI")
axes[0].set_title("Strength vs OPI")

axes[1].scatter(joined["strength_28"], joined["sorptivity_mean"], s=15, color="orange")
axes[1].axhline(10, color="red", linestyle="--")
axes[1].set_xlabel("28-day cube strength (MPa)")
axes[1].set_ylabel("Sorptivity")
axes[1].set_title("Strength vs sorptivity")

plt.tight_layout()
plt.savefig(figures / "10_H7_strength_vs_durability.png", dpi=150)
plt.show()

**Your conclusion for H7:** Is the link strong (correlation above about 0.5) or weak? Careful: most joined
tests are W50/20, so strength and mix are mixed up together. Check if the link is still there inside W50/20 only:

In [ ]:
w50 = joined[joined["mix_id"] == "AFR-W50/20"]
print("W50/20 only -", len(w50), "tests")
print("strength vs OPI:", round(w50["strength_28"].corr(w50["opi_mean"]), 2))
print("strength vs sorptivity:", round(w50["strength_28"].corr(w50["sorptivity_mean"]), 2))

## H8 - Why is the CRCP denser?

**Hypothesis:** the CRCP mix design is simply heavier.

**If true:** the design mass (all the kg per m3 added up) of the CRCP should be clearly higher than the other mixes,
and measured density should be close to the design mass for every mix.

In [ ]:
mixes["design_mass"] = mixes["cement"] + mixes["fly_ash"] + mixes["water"] + mixes["coarse_agg"] + mixes["fine_agg"]
measured = cubes.groupby("mix_id")["density"].mean()

check = mixes.set_index("mix_id")[["design_mass"]].copy()
check["measured_density"] = measured
check["difference"] = check["measured_density"] - check["design_mass"]
print(check.round(0))

**Your conclusion for H8:** If the CRCP design mass is NOT higher, but its measured density is the only one
well ABOVE its design, then the design is not the reason. Things to check outside the data:
red pigment (iron oxide is heavy and is not in our mix design numbers), the real batched mix (batch tickets),
and how the CRCP cubes are made on the road.

## Step 9: Write down your conclusions

For each hypothesis write: **supported / rejected / not sure**, and one sentence why.

1. H1 - low-strength mixes erratic because of many different elements:
2. H2 - wet concrete is weaker (water added on site):
3. H3 - slump is high because some elements need it:
4. H4 - 30/20 and 40/20 dropped because the elements changed:
5. H5 - the season changes the strength:
6. H6 - fly ash slows down the early strength:
7. H7 - stronger concrete is more durable:
8. H8 - the CRCP is denser because its design is heavier:

### Answers - Step 2

1. **H1 - low-strength mixes erratic because of many different elements: REJECTED.**
The spread stays high inside one element: 20/20 is 40.5% overall but still 38.5% in duct encasement and 37.1% in stormwater (only sidewalks are lower at 25.8%). 35/20 is 35.8% in retaining walls and 25/20 stays at about 21-26% everywhere. So the element is not the main reason - the concrete itself, or how it is sampled and cured on small jobs, is erratic. The "class only" results (report just says "20MPa") are higher and even more spread out (20/20: 34.8 MPa, std 14.0 vs 27.6 MPa, std 10.9), so some of them were probably a different mix.

2. **H2 - wet concrete is weaker (water added on site): REJECTED.**
Concrete above the design slump is slightly STRONGER, not weaker: 30/20 42.3 vs 39.7 MPa (p < 0.001), 40/20 54.4 vs 51.6 MPa (p = 0.013), 20/20 31.2 vs 27.3 MPa (p = 0.046). Density is the same for wet and normal concrete. If water was added, both strength and density would drop. The extra slump most likely comes from admixture (plasticiser) in the supplied mix, not water.

3. **H3 - slump is high because some elements need it: PARTLY SUPPORTED.**
For 40/20 the element matters: parapets are too wet 77% of the time vs 38% for other elements (thin, lots of steel, poured wetter). But 30/20 is about 50% too wet in every element (MSE walls 56%, stormwater 54%, piled walls 51%). So the element matters for 40/20, but for 30/20 it is how the mix is supplied - the 75-125 mm range is probably lower than what the supplier and contractor really work with.

4. **H4 - 30/20 and 40/20 dropped because the elements changed: REJECTED.**
The drop is inside the same element: 30/20 in MSE walls went from 44.1 MPa (2024-H1) to 39.6-40.9 MPa (2025-2026). The 7-day strength dropped too (31-33 -> 26-28 MPa), while slump did not go up and density stayed the same (about 2,375-2,395). So the concrete itself changed - same density but lower strength points to the binder (cement / fly ash quality or proportions), not the aggregate. 40/20 shows the same (63.4 MPa in 2024 -> 48-50 MPa in 2026, no rise in slump). This is the most important finding for Step 3.

5. **H5 - the season changes the strength: NOT SUPPORTED (small effect).**
The differences between seasons are mostly below 0.1 of the class and go in different directions for different mixes. Summer is a little lower for W40/20 (1.20 vs 1.27-1.31 at 28 days), and winter 7-day is slightly lower for 30/20 and 40/20 (0.95 vs about 1.0). This fits cubes that are cured in the lab and only feel the weather on the first day.

6. **H6 - fly ash slows down the early strength: SUPPORTED.**
Fly ash mixes reach about 75% of their 28-day strength at 7 days vs 86% for the CRCP without fly ash (p < 0.001). So a 7-day result must be judged differently for fly ash concrete - it looks weaker at 7 days than plain cement concrete of the same 28-day strength. (The CRCP is also a different mix and element, so this is strong evidence, not full proof.)

7. **H7 - stronger concrete is more durable: PARTLY SUPPORTED, weak link.**
Across all pours, higher 28-day strength goes with better OPI (+0.43) and lower sorptivity (-0.32). But inside W50/20 only, the link is much weaker (+0.30 with OPI, -0.08 with sorptivity), so a lot of the overall link comes from comparing different mixes. Strength alone can not predict durability well - and all tests passed anyway.

8. **H8 - the CRCP is denser because its design is heavier: REJECTED.**
The CRCP design mass (about 2,408 kg/m3) is not higher than the others - 30/20 and 40/20 are designed heavier (2,430-2,444). But the CRCP is the mix furthest ABOVE its design (+32 kg/m3), while 30/20 and 40/20 come out about 47 kg/m3 below theirs. The reason is outside our data - most likely the red pigment (heavy, not in our mix numbers) or the real batched mix being different from the comparison sheet we used.